# `C06` — Multi-level walk and volume-weighted price

Scenario **C06** of the Track A certification ladder (card `OMIE-CERT-C06-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C6). Oracle kind: **exact-parametric**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** on order ids, prices in ticks (0.01 EUR/MWh) and quantities in ticks (0.1 MW), and on products and sums of them computed in exact (unbounded) Python integers, **before any division**. Euro figures and the volume-weighted average price itself are shown, never asserted.

## 1. Purpose

**Purpose.** A large order consumes several price levels and prints several trades at different
prices; the resulting average is what the agent actually pays, so this is where execution cost
becomes real. Two failure modes are targeted. The first is level iteration — stopping one level early
or double-counting one. The second is quieter and is why the scenario carries a deliberate stress
draw: the accumulator summing $\text{price}\times\text{quantity}$ runs in integer space and can overflow.
`trade_extraction.py` documents an int32-to-int64 promotion for exactly this reason, so one
parameterisation sits at the ~2 × 10¹⁰ product boundary. The comparison is made on the integer
numerator and denominator **before any division**, so an overflow cannot be masked by rounding. The
trade-ring precondition binds here in particular: a twelve-level sweep repeated across draws is one
of the easiest ways to wrap the 100-trade buffer.

*(Verbatim from `scenario-ladder.md` §C6.)*

**Certifies** (card): §6.2 item 6 (level continuation). The int64 $\text{price}\times\text{quantity}$ accumulator is an **engineering invariant, not a spec rule**, and every assertion that rests on it is labelled `ENGINEERING INVARIANT`.

**How the claim is pinned.** Every draw is one ladder of $L$ resting NON orders on one side of one contract, one order per price level, followed by one NON **aggressor** on the other side. The aggressor's size is the whole of the first $m$ levels plus part of level $m$, and its limit is exactly level $m$'s price. Two constructions:

- `C06.multi_level_walk` — the random ladder: $L \in [3, 12]$, $1 \le m < L$, a partial $0 < r < q_m$ of level $m$ (500 draws).
- `C06.multi_level_walk_stress` — the stress draw at the promotion boundary: one fill of 999 MW at 9,000 EUR/MWh, so that the single product $p \times q = 1{,}899{,}900 \times 9{,}990 = 18{,}980{,}001{,}000 \approx 1.9 \times 10^{10}$, almost nine times the int32 maximum $2^{31}-1 = 2{,}147{,}483{,}647$ (finding F2).

Per draw, separately (section 5): **(Q)** the exact counterparty sequence, one trade row per level touched, best level first, each at that level's own limit; **(V)** the exact per-level fill vector; **(K)** the triple `(total_qty_int, weighted_price_int, num_fills)`, where `weighted_price_int` is the integer **numerator** $W = \sum_i p_i\,q_i$ and `total_qty_int` the integer **denominator** $Q = \sum_i q_i$, compared before any division; **(B)** the book at the end (level $m$'s remainder and the untouched deeper levels, no aggressor residual); **(A)** every order accepted; and **(E)**, an `ENGINEERING INVARIANT`, the engine's own report of the aggressor's submission.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card cites `market-spec.md` v2.12; the file is now v2.13. v2.13 changed §4.2.1 Rule 3 and §6.2 item 7 only (OQ-31), neither of which C06 quotes. The passages in section 2 are checked against the current file at run time.
- **F2 — the stress draw is not a walk.** `C06.multi_level_walk_stress` has $L = 2$, $m = 1$ and no partial: one trade, of the whole first level (999 MW at 9,000 EUR/MWh), with the second level crossable but untouched because the aggressor is used up. It certifies the **single-product** promotion (the per-row $Q_{\text{tick}} \times p_{\text{tick offset}} \approx 2\times10^{10}$ of OQ-18), not a walk; the card's "$L \in [3,12]$ … plus a partial" holds for the random draws only, and section 4's precondition states each construction's own range. A **multi-term** sum past int32 is reached by the random draws only rarely (2 of the first 500 seeds have $W > 2^{31}-1$). A coverage check requires at least one, and such a draw is always also run eager.
- **F3 — the engine's accumulator is not observable as an integer.** The record exposes each trade row's integers and the engine's own report of the submission (`EngineReport`: `trade_qty_mw`, `trade_price_eur`, `settlement_value_eur`, all floats). (K) is therefore **summed by this notebook, in exact Python integers, over the engine's trade rows**: it certifies level iteration and every row's integers exactly, and it cannot itself overflow. The engine's own accumulation is reached only through its report. (E) asserts that the reported quantity, converted to ticks, equals $Q$ exactly (the conversion I6(b) uses). It also asserts that the reported price, converted to ticks, lies within the fill-price bracket $[\min_i p_i - 1, \max_i p_i + 1]$. The spec does not define a reported price, so the bracket holds whether that figure is the VWAP, the first or the last trade price; the one tick of slack absorbs float32 rounding, and an int32 wrap moves the figure far outside the bracket. The settlement value is checked by the battery's I6 within its derived float32 tolerance. Equality of the reported price with $W/Q$ is **reported, not asserted**.
- **F4 — no competition at one price.** Every level holds exactly one resting order at a distinct price, and the tape timestamps strictly increase (section 4's precondition). So the price-time ties C03 certifies, and the same-second tie found by C05's K6-F, do not arise here. The exact counterparty sequence is still pinned (domain review §17.2, D3-08 practice).
- **F5 — the trade ring cannot wrap across draws.** Every draw runs on a fresh book, so one run writes at most $L \le 12$ trade rows; "a twelve-level sweep repeated across draws" does not accumulate in one ring here. The margin is still checked from the tape before every run, and I8 confirms it after.
- **F6 — what `num_fills` counts.** It is the number of trade rows written by the aggressor's submission. One per level touched follows from §6.2 items 3 and 6 with one order per level, and from the matching algorithm's "a trade is always between exactly two orders" (§5.3, shown as a citation in section 2). A level split over two rows would be a double count and fails (Q) and (K).
- **F7 — the judge's own int32 hazard.** The tape is int32. Every field is converted to a Python `int` before any product is formed, because a numpy int32 product would overflow **in the judge**, not in the engine.

## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **Which level first (§6.1).** The resting side is ranked best price first: lowest ask first, highest bid first. With one order per level, the rank order is the level order.
- **Whether a level is reached (§6.1, §6.2 items 1–2).** The aggressor meets the best counterpart while `sell_price ≤ buy_price`. Its limit is level $m$'s price, so levels $0..m$ cross and deeper ones do not.
- **At what price (§6.2 item 3).** Each trade prints at the resting order's limit, so level $i$'s trade prints at $p_i$.
- **Continuation (§6.2 item 6), the rule certified.** When a level is used up, matching continues at the next best level, until the aggressor is used up. No level is skipped and none is consumed twice.
- **What remains (§4.5 NON).** Level $m$, partially executed, stays in the book with $q_m - r$; the aggressor, used up, does not rest.
- **The value (§8.1).** Each trade's value is $Q \times p \times d$ at the level's own price. In ticks the walk's total is the numerator $W = \sum_i p_i\,q_i$ (the tick sizes and $d$ are constant factors, and the price offset is affine), compared exactly together with the denominator $Q = \sum_i q_i$.
- **Bounds (§2.3).** The stress draw's 999 MW at 9,000 EUR/MWh is legal.
- **Engineering, not market rules.** OQ-18 records why $p \times q$ at the OMIE envelope (~$2\times10^{10}$) needs an int64 accumulator, and why a jitted trace can silently lose it. The domain review's §5 item 2 records why the trade-ring margin is checked before the oracle.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §6.1: the ranking that fixes the order the levels are consumed in ------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="BUY-side ladders: the aggressor (a SELL) meets the highest bid first, then the next lower one.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="SELL-side ladders: the aggressor (a BUY) meets the lowest ask first, then the next higher one.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="The walk continues only while the best remaining level still crosses the aggressor's limit.",
    ),
    # --- §6.2: regular matching, item by item -----------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The aggressor is the incoming order; the ladder is the book.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="The crossing test of the reference walk (W).",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Each level's trade prints at that level's own limit, never at the aggressor's.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 6",
        "If matching exhausts one price level, continues to next best price level.",
        note="The rule C06 certifies: level continuation, with no level skipped and none consumed twice.",
    ),
    # --- §4.5: the partially executed last level stays in the book -------------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the NON row below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="Every row is NON: the last level is partially filled and its remainder stays in the book.",
    ),
    # --- §8.1: the per-trade value formula, one trade per level ----------------------------------------
    SpecQuote(
        "market-spec.md", "§8.1",
        "V (EUR) = Q (MW) × p (EUR/MWh) × d (h)",
        note="Per trade. Summed over the walk's trades, in ticks, it is the numerator W = Σ p_i q_i "
             "(d and the tick sizes are constant factors).",
    ),
    SpecQuote(
        "market-spec.md", "§8.1",
        "`p`: trade price in EUR/MWh (resting order's limit price in regular matching; arithmetic mean of "
        "last pair in batch matching)",
        note="Regular matching: p is the level's own limit, so W pairs each level's price with its fill.",
    ),
    # --- §2.3: the bounds the stress draw sits inside --------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price bounds:** −9,999 EUR/MWh (minimum) to +9,999 EUR/MWh (maximum).",
        note="The stress draw prices at 9,000 EUR/MWh, inside the bound (price_int 1,899,900 in the offset "
             "encoding).",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).",
        note="The stress draw fills 999 MW (qty_int 9,990), the largest legal single fill.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price tick (minimum increment):** 0.01 EUR/MWh (2 decimal places maximum).",
        note="The price tick of every price_int.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity tick (minimum increment):** 0.1 MW (1 decimal place maximum).",
        note="The quantity tick of every qty_int.",
    ),
    # --- engineering context, not market rules ----------------------------------------------------------
    SpecQuote(
        "open-questions.md", "OQ-18",
        "the x64 flag is read at trace time, so `int64` casts inside\n"
        "the traced body silently collapse to `int32`, losing the two-term\n"
        "accumulator's overflow guarantees at the OMIE envelope (per-row\n"
        "`Q_tick × p_tick_offset ≈ 2×10¹⁰`, outside int32 range ~2.15×10⁹).",
        block="Context.",
        note="ENGINEERING INVARIANT, not a spec rule: why the stress draw sits at ~2×10¹⁰, and why its eager "
             "and jitted runs are both compared.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§5 item 2",
        "**`nTrades = 100` is the likeliest accidental collision**, and nothing flagged it. A multi-level\n"
        "   walk, an iceberg cascade, a settlement draw or a fuzz session can each exceed 100 trades, and a\n"
        "   wrap silently corrupts I1/I2/I6 — *the very invariants used as the oracle*.",
        note="ENGINEERING precondition: the trade-ring margin is checked before every run, so a wrap discards "
             "a draw instead of failing it.",
    ),
)
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.1 — "
        "docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_"
        "Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md",
        "Two orders are matched at the limit price of the order that was already in the order book.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.3 — same file",
        "If an order with a new timestamp can be executed, it is not necessarily executed at a single price "
        "(except orders with the execution restriction AON ), but may sequentially generate multiple "
        "transactions at different prices against multiple different orders that already existed in the order "
        "book. As soon as the order has been executed against all orders at a certain price limit, the next "
        "best price level becomes best and the order continuous to be matched against orders entered at this "
        "price level. This process continuous as long as the incoming order remains executable and has a "
        "positive order quantity.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.3 — same file",
        "Whenever two orders are matched, a trade is created. Even though one order can match with multiple "
        "other or ders, a trade is always between exactly two orders.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C06.multi_level_walk` (the card's default for a random scenario) and **`STRESS_DRAWS = 4`** of `C06.multi_level_walk_stress`. The card asks for "one stress draw"; four seeds cost a few seconds and cover both resting sides (seeds 1,000,000–1,000,003 rest SELL, BUY, BUY, BUY). The run requests `DRAWS + STRESS_DRAWS` valid draws.
- **`SEED0 = 0`** is the first seed of the walk; **`STRESS_SEED0 = 1_000_000`** the first of the stress draw. The ranges are disjoint, so a seed names one draw only, in the results JSON and in a frozen counterexample. A draw found outside the capacity margin before the run, or marked I8 `invalid` by the battery after it, is **discarded and replaced by the next seed**. Each construction's loop stops at its draw count or after twice that many attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 5`.** The first `EAGER_DRAWS` **valid** walk draws also run eager and are compared, integer field by integer field, with their jitted run (`compare_records`). On top of these, an eager run is **forced** for the first valid draw of every class: walk × resting side, the walk with $L = 12$, the walk with $W > 2^{31}-1$, stress × resting side. **Every** valid stress draw also runs eager (OQ-18: a jitted trace can silently lose an int64 cast). Every one of these **must** have its eager comparison: a missing one is recorded as a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them. (E) is also checked on each eager record, since `compare_records` does not compare the report's euro figures.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros and to convert the engine's float report to ticks in (E).

A run with far fewer draws than the default can miss a class (a resting side, $L = 3$ or $12$, $m = 1$ or $L-1$, $W > 2^{31}-1$). It then fails the coverage check and the missing class's eager comparison. That is intended: such a run does not cover the card's dataset.

In [ ]:
CERT_ID = "C06"            # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws of C06.multi_level_walk (card: random scenarios default to 500 draws)
STRESS_DRAWS = 4           # valid draws of C06.multi_level_walk_stress (card: one stress draw; 4 cover both sides)
SEED0 = 0                  # first seed of C06.multi_level_walk
STRESS_SEED0 = 1_000_000   # first seed of C06.multi_level_walk_stress (a disjoint range: one seed names one draw)
EAGER_DRAWS = 5            # the first EAGER_DRAWS valid walk draws also run eager (plus forced classes, every stress)

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

WALK = "C06.multi_level_walk"
STRESS = "C06.multi_level_walk_stress"
CONSTRUCTIONS = (WALK, STRESS)
N_DRAWS = {WALK: DRAWS, STRESS: STRESS_DRAWS}
FIRST_SEED = {WALK: SEED0, STRESS: STRESS_SEED0}
REQUESTED = DRAWS + STRESS_DRAWS
L_RANGE = {WALK: (3, 12), STRESS: (2, 12)}   # finding F2: the stress draw has L = 2

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)
GFS = int(Validity.GFS)

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks
INT32_MAX = 2 ** 31 - 1               # the promotion boundary (engineering invariant, OQ-18)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C06_multi_level_walk_vwap.ipynb"

print(f"{CERT_ID}: {WALK} x {DRAWS} (seeds from {SEED0}), {STRESS} x {STRESS_DRAWS} (seeds from {STRESS_SEED0}); "
      f"requested {REQUESTED} valid draws, at most 2x attempts per construction")
print(f"eager: first {EAGER_DRAWS} valid walk draws + forced classes + every stress draw")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display and report conversion only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg; int32 max {INT32_MAX:,}")

## 4. Dataset

**`C06.multi_level_walk`** and **`C06.multi_level_walk_stress`** (`omie_tape_synth.make_multi_level_walk`, `stress=False` / `True`) each emit $L + 1$ NON, GFS rows on one contract, order ids $1..L+1$ and tape timestamps $1..L+1$ in row order. Write $\sigma$ for the resting side and $\bar\sigma$ for the other.

| row (submission $s$) | id | side | limit | size |
|---|---|---|---|---|
| level $i$, $s = i$, $i = 0..L-1$ | $i + 1$ | $\sigma$ | $p_i$, strictly worse for a taker as $i$ grows (SELL ascending, BUY descending) | $q_i$ |
| aggressor $a$, $s = L$ | $L + 1$ | $\bar\sigma$ | $p_m$ (level $m$'s price) | $Q_a = \sum_{i<m} q_i + r$ |

**What section 5 reads.** The expectation reads the **tape rows**: their order, sides, prices, sizes and order ids, through one reference walk. `draw.params` gives the closed form, which is cross-checked against the walk on every draw and checked against the tape. `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate assertion in section 7):

- **Shape.** $L+1$ real rows, all NON, GFS, no peak, one contract. Order ids $1..L+1$ equal `params`' `resting_oids` + `agg_oid`; tape timestamps strictly increasing; prices and sizes within §2.3's bounds.
- **Sides and values.** The first $L$ rows on `rest_side`, the last on the other side (`agg_side`). `level_price_ints`, `level_qty_ints`, `p_agg_int` and `q_agg_int` equal the tape's values.
- **One order per level.** The level prices are strictly monotone in the taker-worse direction, so each level holds one order and the tape order is the rank order (finding F4).
- **The walk.** $L$ within the construction's range ($[3,12]$ walk, $[2,12]$ stress: finding F2); $1 \le m < L$; $0 \le r < q_m$ with $r > 0$ for the walk (the "plus a partial"); $Q_a = \sum_{i<m} q_i + r$; $p_a = p_m$.
- **The stress.** On every stress draw the largest single product $p_i\,q_i$ of the expected fills exceeds $2^{31}-1$.
- **No icebergs.** None (so $\lceil qty/peak\rceil$ is 0).

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). A ladder of at most 12 levels plus one aggressor sits far inside it: at most 12 resting per side, at most 13 trade units, a hop bound of at most 15. It is still checked **before** each run: a tape whose analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), or whose largest $\lceil qty/peak \rceil$ exceeds 64, is never run; the draw is discarded and re-drawn. After the run, the battery's I8 confirms the flags.

Below: the first seed of each construction in readable units beside the integers, the book shape, and the dataset's class counts.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_VALIDITY,
                                             synth.COL_PEAK)


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.1 / §6.2 item 2: sell price <= buy price, for a resting order on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((-(-int(r[C_QTY]) // max(1, int(r[C_PEAK]))) for r in ice), default=0)


DRAWN = {}


def prepare(key, seed):
    # Draw one seed of one construction (the run loop of section 7 calls this for a re-draw too).
    d = synth.CONSTRUCTIONS[key](seed)
    DRAWN[(key, seed)] = d
    return d


for _key in CONSTRUCTIONS:
    for _s in range(FIRST_SEED[_key], FIRST_SEED[_key] + N_DRAWS[_key]):
        prepare(_key, _s)


def tape_table(tape, title):
    rows = ["| s | role | oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | tape ts | exec |",
            "|---|---|---|---|---|---|---|---|---|---|---|"]
    real = real_rows(tape)
    for k, r in enumerate(real):
        role = "aggressor" if k == len(real) - 1 else f"level {k}"
        rows.append(f"| {k} | {role} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} | "
                    f"{'NON' if int(r[C_EXEC]) == NON else int(r[C_EXEC])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


for _key in CONSTRUCTIONS:
    _s = FIRST_SEED[_key]
    _d = DRAWN[(_key, _s)]
    display(Markdown(tape_table(_d.tape, f"{_key}, seed {_s} (digest `{_d.digest()[:16]}…`)")))
    print(f"{_key} seed {_s} params: {_d.params}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, _key in zip(axes, CONSTRUCTIONS):
    _d = DRAWN[(_key, FIRST_SEED[_key])]
    _real = real_rows(_d.tape)
    for k, r in enumerate(_real[:-1]):
        buy = int(r[C_SIDE]) == BUY
        q = decode_qty(int(r[C_QTY]), CONFIG)
        p = decode_price(int(r[C_PRICE]), CONFIG)
        ax.barh(p, q * (1 if buy else -1), height=0.6 * max(CONFIG.price_tick, abs(float(_d.params["step_eur"]))),
                color="tab:green" if buy else "tab:red")
        ax.annotate(f"L{k}", (q * (1 if buy else -1), p), fontsize=7, va="center")
    _a = _real[-1]
    ax.axhline(decode_price(int(_a[C_PRICE]), CONFIG), color="black", ls="--", lw=0.8,
               label=f"aggressor {side_name(_a[C_SIDE])} limit, {decode_qty(int(_a[C_QTY]), CONFIG):.1f} MW")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("resting quantity (MW): buy right, sell left")
    ax.set_ylabel("limit (EUR/MWh)")
    ax.set_title(f"{_key}, seed {FIRST_SEED[_key]}: the ladder and the aggressor's limit", fontsize=9)
    ax.legend(loc="best", fontsize=7)
plt.tight_layout()
plt.show()

_classes = Counter()
for (_key, _s), _d in DRAWN.items():
    _r = real_rows(_d.tape)
    _classes[(_key, f"resting {side_name(_r[0][C_SIDE])}")] += 1
    _classes[(_key, f"L = {len(_r) - 1}")] += 1
display(Markdown("**Classes in the initial seed range** (counts; $m$ and $W$ classes are in section 5)\n\n"
                 "| construction | class | draws |\n|---|---|---|\n"
                 + "\n".join(f"| {k} | {c} | {n} |" for (k, c), n in sorted(
                     _classes.items(), key=lambda kv: (kv[0][0], len(kv[0][1]), kv[0][1])))))

## 5. Known result

**Notation.** Integer ticks throughout, and every product in exact (unbounded) Python integers (finding F7). The tape's real rows, in submission order $s = 0..L$, are levels $0..L-1$ and then the aggressor $a$ (section 4). Every row is NON on one contract.

**(W) The reference walk** (§6.1, §6.2 items 1–3 and 6, §4.5). Process the rows in submission order. An entering order with limit $p$ and size $q$ meets the opposite side ranked by $(p_e, s_e)$ (SELL resting: lowest price first) or $(-p_e, s_e)$ (BUY resting: highest price first), where $s_e$ is the entry's submission index. While it has quantity left and the best opposite entry crosses (§6.2 item 2), it trades $f = \min(\text{left}, q_e)$ **at $p_e$** (§6.2 item 3), and an entry used up leaves the book (§6.2 item 6: the next level becomes best). Whatever is left of the incoming order rests at its own limit (§4.5 NON). The ladder rows meet an empty or same-side book and rest whole.

**Closed form** (from `draw.params`: level prices $p_i$, sizes $q_i$, order ids $o_i$, $m$, the partial $r$ = `partial_qty_int`, the aggressor $a$). With $\mathbb{1}[r>0]$ equal to 1 when there is a partial:

$$\text{fills} = \big[\,[o_i,\, a,\, p_i,\, q_i,\, L]\ \text{for } i = 0..m-1\,\big] \;+\; \big[\,[o_m,\, a,\, p_m,\, r,\, L]\ \text{if } r > 0\,\big]$$

$$v = (q_0, \dots, q_{m-1},\; r,\; 0, \dots, 0) \in \mathbb{Z}^{L}$$

$$\texttt{total\_qty\_int} = Q = \sum_{i<m} q_i + r = Q_a, \qquad \texttt{weighted\_price\_int} = W = \sum_{i<m} p_i\,q_i + p_m\,r, \qquad \texttt{num\_fills} = m + \mathbb{1}[r>0]$$

$$\text{book at the end:}\quad \sigma:\ [p_m,\, q_m - r,\, o_m],\ [p_i,\, q_i,\, o_i]\ \text{for } i = m+1..L-1; \qquad \bar\sigma:\ \text{empty}$$

The volume-weighted price $W/Q$ (in ticks; $W/Q \cdot 0.01 - 9{,}999$ in EUR/MWh) is **shown, never asserted**: $W$ and $Q$ are compared exactly, before any division.

**What is asserted, per valid draw** (each with its expected and observed values recorded in full):

- **(A) acceptance** — every order id $1..L+1$ accepted, none rejected.
- **(Q) counterparty sequence** — the exact fill list above, `[passive, aggressor, price_int, qty_int, submission]`, in trade-row order: levels in rank order, each at its own limit, the last one partial.
- **(V) per-level fill vector** — $v$, indexed by level, best first: a level skipped, stopped short or consumed twice changes it.
- **(K) the triple** — `[total_qty_int, weighted_price_int, num_fills]` $= [Q, W, m + \mathbb{1}[r>0]]$, summed over the engine's trade rows of the aggressor's submission in exact integers.
- **(B) the book at the end** — `[price_int, qty_int, oid]` per side.
- **(E) ENGINEERING INVARIANT, not a spec rule** (finding F3) — the engine's report of submission $L$ says it traded; its quantity, $\text{round}(\texttt{trade\_qty\_mw}/0.1)$, equals $Q$ exactly; its price, $\text{round}((\texttt{trade\_price\_eur} + 9{,}999)/0.01)$, lies in $[\min_i p_i - 1,\ \max_i p_i + 1]$ over the expected fills. It is checked on the jitted record and on every eager record.
- **Battery** — RB and I1–I9 on the jitted record (`expects_trade=True`). I2 closes the quantities and I6 the settlement value; I8 `invalid` discards the draw.

The code below implements (W) as one function of the tape rows, and the closed form as one function of `draw.params`; section 7 checks that they agree on every draw. Neither reads `draw.expected`, and neither runs the engine.

In [ ]:
def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def book_view(entries):
    # {side: sorted [price_int, qty_int, oid]} of the walk's resting entries.
    out = {"BUY": [], "SELL": []}
    for e in entries:
        out[side_name(e["side"])].append([e["price"], e["qty"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def walk(tape):
    # (W): the reference walk over the real rows, in exact Python integers.
    book, fills = [], []
    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID])
        left = q
        while left > 0:
            opp = [e for e in book if e["side"] != side and crosses(e["side"], e["price"], p)]
            if not opp:
                break
            best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["entry"]))
            f = min(left, best["qty"])
            fills.append([best["oid"], oid, best["price"], f, s])
            best["qty"] -= f
            left -= f
            if best["qty"] == 0:
                book.remove(best)
        if left > 0:
            book.append({"side": side, "price": p, "qty": left, "oid": oid, "entry": s})
    return {"fills": fills, "book_final": book_view(book)}


def ladder_oids(tape):
    # The resting rows (all but the last), ranked best first by §6.1: their order ids.
    rows = list(enumerate(real_rows(tape)))[:-1]
    ranked = sorted(rows, key=lambda kr: rank_key(kr[1][C_SIDE], kr[1][C_PRICE], kr[0]))
    return [int(r[C_OID]) for _, r in ranked]


def triple_of(fills):
    # [total_qty_int, weighted_price_int, num_fills] of a fill list, in exact Python integers.
    return [sum(int(f[3]) for f in fills), sum(int(f[2]) * int(f[3]) for f in fills), len(fills)]


def expectation(tape):
    # Everything section 7 compares, from (W) on the tape rows. None when the tape has no aggressor.
    rows = real_rows(tape)
    if len(rows) < 2:
        return None
    w = walk(tape)
    agg_sub = len(rows) - 1
    agg_fills = [f for f in w["fills"] if f[4] == agg_sub]
    lad = ladder_oids(tape)
    prices = [f[2] for f in agg_fills]
    return {
        "oids": sorted(int(r[C_OID]) for r in rows), "agg_oid": int(rows[-1][C_OID]), "agg_sub": agg_sub,
        "ladder_oids": lad, "fills": w["fills"],
        "vector": [sum(f[3] for f in agg_fills if f[0] == o) for o in lad],
        "triple": triple_of(agg_fills),
        "book_final": w["book_final"],
        "bracket": [min(prices), max(prices)] if prices else None,
        "max_product": max((int(f[2]) * int(f[3]) for f in agg_fills), default=0),
    }


def closed_form(p):
    # The closed form of the formulas above, from draw.params only.
    P = [int(x) for x in p["level_price_ints"]]
    Q = [int(x) for x in p["level_qty_ints"]]
    o = [int(x) for x in p["resting_oids"]]
    a, L, m, r = int(p["agg_oid"]), int(p["L"]), int(p["m"]), int(p["partial_qty_int"])
    fills = [[o[i], a, P[i], Q[i], L] for i in range(m)] + ([[o[m], a, P[m], r, L]] if r > 0 else [])
    rest_left = ([[P[m], Q[m] - r, o[m]]] if Q[m] - r > 0 else []) + [[P[i], Q[i], o[i]] for i in range(m + 1, L)]
    return {
        "fills": fills,
        "vector": Q[:m] + [r] + [0] * (L - m - 1),
        "triple": [sum(Q[:m]) + r, sum(P[i] * Q[i] for i in range(m)) + P[m] * r, m + (1 if r > 0 else 0)],
        "book_final": {side_name(p["rest_side"]): sorted(rest_left), side_name(p["agg_side"]): []},
    }


def closed_form_agrees(draw):
    try:
        cf, e = closed_form(draw.params), expectation(draw.tape)
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False
    return e is not None and all(e[k] == cf[k] for k in cf)


def precondition(key, draw):
    # Section 4's dataset precondition, from the tape and params. Returns the list of departures.
    bad, t, p = [], draw.tape, draw.params
    rows = real_rows(t)
    L = int(p.get("L", -1))
    lo, hi = L_RANGE[key]
    if not lo <= L <= hi:
        bad.append(f"L = {L} outside [{lo}, {hi}]")
    if len(rows) != L + 1 or len(rows) < 2:
        return bad + [f"{len(rows)} real rows (expected L + 1 = {L + 1})"]
    rest, agg = rows[:-1], rows[-1]
    if any(int(x[C_EXEC]) != NON for x in rows):
        bad.append("a row is not NON")
    if any(int(x[C_VALIDITY]) != GFS for x in rows) or any(int(x[C_PEAK]) != 0 for x in rows):
        bad.append("a row is not GFS or carries a peak")
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids = [int(x[C_OID]) for x in rows]
    if oids != list(range(1, L + 2)) or oids != [int(x) for x in p.get("resting_oids", [])] + [int(p.get("agg_oid", -1))]:
        bad.append(f"order ids {oids} vs params")
    ts = [int(x[C_TS]) for x in rows]
    if any(b_ <= a_ for a_, b_ in zip(ts[:-1], ts[1:])):
        bad.append(f"tape timestamps not strictly increasing: {ts}")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX
           for x in rows):
        bad.append("price or quantity outside §2.3")
    rs = int(p.get("rest_side", -1))
    if any(int(x[C_SIDE]) != rs for x in rest) or int(agg[C_SIDE]) != other(rs) or int(p.get("agg_side", -1)) != other(rs):
        bad.append("sides")
    P = [int(x[C_PRICE]) for x in rest]
    Q = [int(x[C_QTY]) for x in rest]
    if P != [int(x) for x in p.get("level_price_ints", [])] or Q != [int(x) for x in p.get("level_qty_ints", [])]:
        bad.append("params level_price_ints / level_qty_ints != tape")
    if int(p.get("p_agg_int", -1)) != int(agg[C_PRICE]) or int(p.get("q_agg_int", -1)) != int(agg[C_QTY]):
        bad.append("params p_agg_int / q_agg_int != tape")
    worse = (lambda x, y: y > x) if rs == SELL else (lambda x, y: y < x)
    if not all(worse(x, y) for x, y in zip(P[:-1], P[1:])):
        bad.append("level prices not strictly monotone (taker-worse): more than one order per level, or out of rank order")
    m, r = int(p.get("m", -1)), int(p.get("partial_qty_int", -1))
    if not 1 <= m < L:
        bad.append(f"m = {m} outside [1, L - 1]")
        return bad
    if not 0 <= r < Q[m] or (key == WALK and r == 0):
        bad.append(f"partial r = {r} not in {'(0' if key == WALK else '[0'}, q_m = {Q[m]})")
    if int(agg[C_QTY]) != sum(Q[:m]) + r:
        bad.append(f"Q_a = {int(agg[C_QTY])} != sum(q_i, i < m) + r = {sum(Q[:m]) + r}")
    if int(agg[C_PRICE]) != P[m]:
        bad.append(f"p_a = {int(agg[C_PRICE])} != p_m = {P[m]}")
    if key == STRESS:
        prods = [P[i] * Q[i] for i in range(m)] + ([P[m] * r] if r > 0 else [])
        if not prods or max(prods) <= INT32_MAX:
            bad.append(f"largest single product {max(prods, default=0)} <= int32 max {INT32_MAX}")
    if max_slices(t) != 0:
        bad.append("an iceberg row")
    return bad


def vwap_eur(triple):
    # Report only: W / Q in EUR/MWh (never asserted).
    Q, W, _ = triple
    return None if Q == 0 else (W / Q) * CONFIG.price_tick + CONFIG.price_min


# ---- the expectation on the first seed of each construction, and the classes over the initial range ----
CF_AGREES = True
for _key in CONSTRUCTIONS:
    _s = FIRST_SEED[_key]
    _d = DRAWN[(_key, _s)]
    _e = expectation(_d.tape)
    _ok = closed_form_agrees(_d)
    CF_AGREES &= _ok
    print(f"{_key} seed {_s}: L = {_d.params['L']}, m = {_d.params['m']}, r = {_d.params['partial_qty_int']}")
    print(f"    (Q) fills {_e['fills']}")
    print(f"    (V) per-level vector {_e['vector']}")
    print(f"    (K) [total_qty_int, weighted_price_int, num_fills] = {_e['triple']} "
          f"(W > int32 max: {_e['triple'][1] > INT32_MAX}; largest single product {_e['max_product']:,})")
    print(f"    (B) book at the end {_e['book_final']}")
    print(f"    (E) fill-price bracket {_e['bracket']}; VWAP shown, not asserted: {vwap_eur(_e['triple']):.4f} EUR/MWh")
    print(f"    closed form == walk: {_ok}; precondition departures: {precondition(_key, _d)}")
print(f"closed form agrees with (W) on the first seeds: {CF_AGREES}")

_mcls = Counter()
for (_key, _s), _d in DRAWN.items():
    _p, _e = _d.params, expectation(_d.tape)
    _mcls[(_key, "m = 1")] += int(_p["m"] == 1)
    _mcls[(_key, "m = L - 1")] += int(_p["m"] == _p["L"] - 1)
    _mcls[(_key, "W > int32 max")] += int(_e["triple"][1] > INT32_MAX)
    _mcls[(_key, "largest single product > int32 max")] += int(_e["max_product"] > INT32_MAX)
display(Markdown("**Walk classes in the initial seed range** (counts)\n\n| construction | class | draws |\n|---|---|---|\n"
                 + "\n".join(f"| {k} | {c} | {n} |" for (k, c), n in sorted(_mcls.items()))))

## 6. Run

### Eager and jitted (wired at K3)

> **K2 STUB — section 6 is not wired yet.** **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, *, jit) -> RunRecord`**: the battery's record builder on one tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, jit=jit)`. It runs on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no cancellations (`cancels_before={}`), no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`, which equal `draw.capacity_bounds` on these draws). `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. The tape is passed **as is** (`draw.tape`): nothing in it may be rewritten.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `passive_oid`, `aggressor_oid`, `price_int`, `qty_int`, `submission` and `row_index`;
- `accepted` (its keys) and `rejections` (each `Rejection.oid`);
- `final_snapshot.resting(side)` (price, qty, oid, …);
- `engine_reports`: the `EngineReport` whose `submission` is the aggressor's ($L$), its `traded`, `trade_qty_mw` and `trade_price_eur`;
- `observed_capacity`;
- the whole record, through `run_battery` and `compare_records`.

Section 7 calls `run_tape(..., jit=True)` on every draw, and `run_tape(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(key, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs (OMIEProductConfig(), OMIEMarketConfig()), no scheduled cancellations, no
# gate-closure expiry and the default analytic capacity bounds (tape_capacity_bounds(tape),
# equal to draw.capacity_bounds on these draws). jit=False wraps the whole run in
# jax.disable_jit() (inside build_record); jit=True runs it jitted. The tape is passed as
# is (draw.tape): no column is rewritten.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, *, jit):
    """One run of `tape` (a draw's tape, unmodified) on a fresh OPEN book."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before={},
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"no cancels, no GCT expiry, default capacity bounds")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks. Then come the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw**, each with its expected and observed value **in full**, so an auditor can check every integer from the results JSON alone: (A) accepted and rejected order ids; (Q) the counterparty sequence; (V) the per-level fill vector; (K) the triple; (B) the final book; and (E) the engine report converted to ticks, beside its raw floats and the bracket. Each per-draw value is recorded once: (V) and (K) are short summaries of (Q), and section 5's closed form is not repeated, only its agreement.

**Aggregate assertions:** the section-6 wiring; the closed form against (W), on the first seeds and on every valid draw; the dataset precondition per construction (section 4), listing every departing seed and its departures; **coverage** — the walk covers both resting sides, $L = 3$ and $L = 12$, $m = 1$ and $m = L - 1$, and at least one $W > 2^{31}-1$; the stress covers both resting sides; and the eager comparisons, one per required class and one per stress draw, a missing one recorded as a failure.

The battery runs on every draw's jitted record with `expects_trade=True`; a draw it marks I8 `invalid` is discarded and replaced by the next seed. **Reported, not asserted:** the engine's reported price against $W/Q$ (finding F3), the largest numerators reached, and the analytic hop-bound peak.

In [ ]:
import math
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Multi-level walk and volume-weighted price",
    known_result="the aggressor walks levels 0..m in rank order, one trade per level at that level's own limit "
                 "(levels 0..m-1 whole, level m for r), never beyond; exact per-level fill vector "
                 "(q_0..q_{m-1}, r, 0..0); [total_qty_int, weighted_price_int, num_fills] = "
                 "[sum q_i + r, sum p_i q_i + p_m r, m + 1[r>0]] in exact integers before any division; "
                 "level m keeps q_m - r, deeper levels whole, no aggressor residual; ENGINEERING INVARIANT: the "
                 "engine's report quantity equals Q and its price lies in the fill-price bracket +-1 tick, "
                 "including at the ~1.9e10 single-product stress draw",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=CONSTRUCTIONS,
    requested_draws=REQUESTED, runtime_forecast_s=120.0, started_at=T0,
    downstream="every scenario where an aggressor meets more than one resting order: C07 (IOC partial and full "
               "regimes walk levels), C08-C10 (multi-slice and ICEwPI walks), C10 (FOK pre-check over levels), "
               "C12 (settlement value of a multi-trade submission, the same price x quantity accumulator), C15 "
               "(random sessions); Track B execution cost (the agent's average fill price) and any reward built "
               "on the engine's reported trade price or settlement value",
)

R_ACC = "§5.1 SUBMITTED -> ACCEPTED; §2.3"
R_SEQ = "§6.1 ranking; §6.2 items 1-3 and 6 (best counterpart, crossing, maker price, level continuation)"
R_VEC = "§6.2 item 6 (level continuation)"
R_TRIPLE = "§6.2 items 3 and 6; §8.1 (per-trade value, summed in integer ticks before any division)"
R_BOOK = "§4.5 NON; §6.2 item 6"
R_ENG = "ENGINEERING INVARIANT, not a spec rule: int64 price x quantity accumulator (OQ-18); I6(b) conversion"
R_DATA = "dataset precondition (section 4); §2.3"


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ACC)
        return None


def to_ticks(x, origin, tick):
    # A float report figure in ticks (round to nearest); None when it is not a finite number.
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return int(round((v - float(origin)) / float(tick))) if math.isfinite(v) else None


def observe_report(rec, e):
    # (E): the engine's report of the aggressor's submission, converted to ticks beside its raw floats.
    er = next((r for r in rec.engine_reports if int(r.submission) == e["agg_sub"]), None)
    if er is None:
        return None
    return {"traded": bool(er.traded),
            "qty_int": to_ticks(er.trade_qty_mw, 0.0, CONFIG.qty_tick),
            "price_int": to_ticks(er.trade_price_eur, CONFIG.price_min, CONFIG.price_tick),
            "trade_qty_mw": float(er.trade_qty_mw), "trade_price_eur": float(er.trade_price_eur)}


def check_report(label, seed, rec, e):
    # (E), ENGINEERING INVARIANT: reported quantity == Q exactly, reported price within the bracket +-1 tick.
    rep = observe_report(rec, e)
    lo, hi = (e["bracket"][0] - 1, e["bracket"][1] + 1) if e["bracket"] else (None, None)
    ok = (rep is not None and rep["traded"] and rep["qty_int"] == e["triple"][0] and rep["price_int"] is not None
          and lo is not None and lo <= rep["price_int"] <= hi)
    cert.check(f"{label}: (E) ENGINEERING INVARIANT: engine report of submission {e['agg_sub']}: traded, "
               f"quantity == total_qty_int, price_int within the fill-price bracket +-1 tick", ok, seed=seed,
               expected={"traded": True, "qty_int": e["triple"][0], "price_int_in": [lo, hi]}, observed=rep,
               unit="other", rule=R_ENG)
    return rep, bool(ok)


def observe(rec, e):
    # What the engine did on one run, in the expectation's terms.
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    seq = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
           for f in fills]
    agg = [x for x in seq if x[4] == e["agg_sub"]]
    fin = rec.final_snapshot
    return {
        "accepted": sorted(int(o) for o in rec.accepted), "rejected": sorted(int(r.oid) for r in rec.rejections),
        "seq": seq,
        "vector": [sum(x[3] for x in agg if x[0] == o) for o in e["ladder_oids"]],
        "triple": triple_of(agg),
        "book_final": {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in fin.resting(sd))
                       for sd in (BUY, SELL)},
    }


def check_run(label, seed, rec, e):
    # (A), (Q), (V), (K), (B), (E) of section 5. Returns (observation, report, all passed).
    o = observe(rec, e)
    ok = cert.check_equal(f"{label}: (A) accepted order ids / rejected order ids", [o["accepted"], o["rejected"]],
                          [e["oids"], []], seed=seed, unit="other", rule=R_ACC)
    ok &= cert.check_equal(f"{label}: (Q) counterparty sequence [passive, aggressor, price_int, qty_int, "
                           f"submission]", o["seq"], e["fills"], seed=seed, unit="other", rule=R_SEQ)
    ok &= cert.check_equal(f"{label}: (V) per-level fill vector, best level first",
                           o["vector"], e["vector"], seed=seed, unit="qty_int", rule=R_VEC)
    ok &= cert.check_equal(f"{label}: (K) [total_qty_int, weighted_price_int, num_fills]", o["triple"],
                           e["triple"], seed=seed, unit="other", rule=R_TRIPLE)
    ok &= cert.check_equal(f"{label}: (B) book at the end [price_int, qty_int, oid]", o["book_final"],
                           e["book_final"], seed=seed, unit="other", rule=R_BOOK)
    rep, rep_ok = check_report(label, seed, rec, e)
    return o, rep, bool(ok and rep_ok)


EAGER_DONE = {}


def eager_compare(label, seed, tape, rec, tags, e):
    # Run `tape` eager, compare with its jitted record (one eager_jit assertion), and check (E) on the eager record.
    rec_e = guarded(f"{label} (eager)", seed, run_tape, tape, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    for t in tags:
        EAGER_DONE.setdefault(t, seed)
    cert.set_eager_jit(seed, diff == [], detail=f"{label}: " + (", ".join(diff) if diff else "identical"))
    if rec_e is not None:
        check_report(f"{label} (eager)", seed, rec_e, e)
    return diff == []


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = [f"{k} bound {bounds.get(k)} > {m}" for k, m in lim.items() if int(bounds.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on the first seeds", CF_AGREES, expected=True,
           observed=CF_AGREES, unit="bool", rule=R_SEQ)
wired = callable(globals().get("run_tape"))
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_ACC)

PER_RUN, DATA_BAD, CLOSED_BAD = [], defaultdict(dict), []
COVER, VALID_SEEDS = defaultdict(Counter), defaultdict(list)
PEAK = {"hop_bound": 0, "max_slices": 0}
NUMERATORS = {WALK: [], STRESS: []}   # (seed, W, largest single product) per valid draw: reported
REPORTS = []                          # (key, seed, W/Q in ticks, reported price_int): reported, not asserted


FORCED = {WALK: ("resting BUY", "resting SELL", "L = 12", "W > int32 max"),   # forced-eager classes
          STRESS: ("resting BUY", "resting SELL")}


def classes(key, draw, e):
    # The coverage classes of one valid draw (also the forced-eager tags).
    p = draw.params
    side = side_name(p["rest_side"])
    tags = [(key, f"resting {side}")]
    if key == WALK:
        L, m = int(p["L"]), int(p["m"])
        tags += [(key, f"L = {L}")] if L in (3, 12) else []
        tags += [(key, "m = 1")] if m == 1 else []
        tags += [(key, "m = L - 1")] if m == L - 1 else []
        tags += [(key, "W > int32 max")] if e["triple"][1] > INT32_MAX else []
    return tags


def run_construction(key):
    first, n = FIRST_SEED[key], N_DRAWS[key]
    valid, seed = 0, first
    while valid < n and seed < first + 2 * n:
        draw = DRAWN.get((key, seed)) or prepare(key, seed)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{key}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        bad = precondition(key, draw)
        e = expectation(draw.tape)
        if bad or e is None:
            DATA_BAD[key][seed] = bad or ["expectation undefined"]
            cert.check(f"{key} seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[],
                       observed=DATA_BAD[key][seed], unit="other", rule=R_DATA)
            seed += 1
            continue
        lbl = f"{key} seed {seed}"
        rec = guarded(f"{lbl} (jit)", seed, run_tape, draw.tape, jit=True)
        report = run_battery(rec, expects_trade=True) if rec is not None else None
        if report is not None and not cert.add_battery(report, seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed
            continue
        valid += 1
        VALID_SEEDS[key].append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": key})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        NUMERATORS[key].append((seed, e["triple"][1], e["max_product"]))
        tags = classes(key, draw, e)
        for t in tags:
            COVER[key][t[1]] += 1
        if not closed_form_agrees(draw):
            CLOSED_BAD.append(f"{key} seed {seed}")
        if rec is not None:
            cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            o, rep, ok = check_run(lbl, seed, rec, e)
            PER_RUN.append({"key": key, "seed": seed, "ok": ok})
            if rep is not None and e["triple"][0]:
                REPORTS.append((key, seed, e["triple"][1] / e["triple"][0], rep["price_int"]))
        if key == STRESS:
            tags = tags + [(key, f"seed {seed}")]
        if key == STRESS or valid <= EAGER_DRAWS or any(t not in EAGER_DONE for t in tags if t[1] in FORCED[key]):
            eager_compare(lbl, seed, draw.tape, rec, tags, e)
        seed += 1
    return valid


VALID = {}
if wired:
    for _key in CONSTRUCTIONS:
        VALID[_key] = run_construction(_key)

# ---- aggregate assertions (every list in full) ----------------------------------------------------------
cert.check("section 5: closed form agrees with the walk (W) on every valid draw", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_SEQ)
for _key in CONSTRUCTIONS:
    _bad = DATA_BAD.get(_key, {})
    cert.check(f"dataset: every {_key} draw is as section 4 describes (precondition)", wired and not _bad,
               expected={}, observed={str(k): v for k, v in _bad.items()}, unit="other",
               detail=f"{len(_bad)} draws depart" if _bad else f"{VALID.get(_key, 0)} valid draws", rule=R_DATA)
_WANT = {WALK: ("resting BUY", "resting SELL", "L = 3", "L = 12", "m = 1", "m = L - 1", "W > int32 max"),
         STRESS: ("resting BUY", "resting SELL")}
for _key in CONSTRUCTIONS:
    _got = {c: COVER[_key][c] for c in _WANT[_key]}
    _missing = [c for c, v in _got.items() if v == 0]
    cert.check(f"dataset: {_key} covers {', '.join(_WANT[_key])}", wired and not _missing,
               expected={c: ">= 1" for c in _WANT[_key]}, observed=_got, unit="count",
               detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- eager: every required class, and every stress draw, must have had one comparison (absence = failure) ----
REQUIRED_EAGER = [(k, c) for k in CONSTRUCTIONS for c in FORCED[k]] + [(STRESS, f"seed {s}") for s in VALID_SEEDS[STRESS]]
for _tag in REQUIRED_EAGER:
    if _tag not in EAGER_DONE:
        cert.check(f"eager == jitted for class {_tag}", False, kind="eager_jit", expected=True, observed=None,
                   unit="bool", detail=f"no eager comparison was run for required class {_tag}")

# ---- reports (not asserted) ----------------------------------------------------------------------------------
_fail_runs = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_RUN)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"{r['key']} seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws per construction: {VALID}; eager comparisons run for classes: {sorted(EAGER_DONE)}")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']}, max ceil(qty/peak) {PEAK['max_slices']}")
for _key in CONSTRUCTIONS:
    if NUMERATORS[_key]:
        _s, _W, _mp = max(NUMERATORS[_key], key=lambda x: x[1])
        print(f"{_key}: largest numerator W = {_W:,} (seed {_s}; largest single product {_mp:,}); "
              f"draws with W > int32 max: {[s for s, W, _ in NUMERATORS[_key] if W > INT32_MAX]}")
if REPORTS:
    _dev = [(k, s, (rp - wq) if rp is not None else None) for k, s, wq, rp in REPORTS]
    _num = [abs(d) for _, _, d in _dev if d is not None]
    print(f"reported, not asserted (finding F3): engine report price_int - W/Q over {len(_dev)} jitted runs: "
          f"max |deviation| {max(_num) if _num else None} ticks; unconvertible reports: "
          f"{[(k, s) for k, s, d in _dev if d is None]}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))